# 04 — Modeling

Train a majority-class baseline plus interpretable (logistic regression, decision tree) and ML (random forest, gradient boosting) classifiers. Compare them on accuracy, precision, recall, F1, and ROC-AUC.

**Inputs**: `data/processed/cleaned.csv`  
**Outputs**: results table; optionally serialized models in `models/`.


In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score

from src.config import PROCESSED_DIR, TARGET, RANDOM_STATE
from src.features.build import build_preprocessor
from src.models.train import candidate_models, make_pipeline
from src.models.evaluate import compare

In [3]:
# load the cleaned data and split it into features (X) and target (y)
df = pd.read_csv(PROCESSED_DIR / 'cleaned.csv')
y = df[TARGET]
X = df.drop(columns=[TARGET])
numeric = X.select_dtypes('number').columns.tolist()
categorical = X.select_dtypes(exclude='number').columns.tolist()
print('n_numeric:', len(numeric), '| n_categorical:', len(categorical))

n_numeric: 12 | n_categorical: 2


In [4]:
# hold out 20% for testing, keeping the same class balance in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print('train:', X_train.shape, '| test:', X_test.shape)

train: (2400, 14) | test: (600, 14)


In [5]:
# build the preprocessing once, then train every model on the training set
preprocessor = build_preprocessor(numeric, categorical)
fitted = {}
for name, model in candidate_models().items():
    pipe = make_pipeline(preprocessor, model)
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
list(fitted.keys())

['baseline',
 'logistic_regression',
 'decision_tree',
 'random_forest',
 'gradient_boosting']

In [6]:
# score every model on the test set and rank them (the key number is ROC-AUC vs the baseline)
results = compare(fitted, X_test, y_test)
results

,accuracy,precision,recall,f1,roc_auc
logistic_regression,0.626667,0.568047,0.388664,0.461538,0.689991
random_forest,0.625000,0.563953,0.392713,0.463007,0.674032
gradient_boosting,0.605000,0.526882,0.396761,0.452656,0.663692
decision_tree,0.585000,0.496212,0.530364,0.512720,0.576797
baseline,0.588333,0.000000,0.000000,0.000000,0.500000


In [7]:
# sanity check with 5-fold cross-validation on the training set to watch for overfitting
cv_scores = {}
for name, pipe in fitted.items():
    if name == 'baseline':
        continue
    scores = cross_val_score(pipe, X_train, y_train, scoring='roc_auc', cv=5, n_jobs=-1)
    cv_scores[name] = {'mean_auc': scores.mean(), 'std_auc': scores.std()}
pd.DataFrame(cv_scores).T.sort_values('mean_auc', ascending=False)

,mean_auc,std_auc
random_forest,0.682807,0.026427
gradient_boosting,0.677500,0.024013
logistic_regression,0.674365,0.033495
decision_tree,0.559290,0.023670


## Notes

- The headline number is ROC-AUC vs. the baseline. Models that don't beat the baseline are not predictive.
- For the interpretable models, also report coefficients (`logistic_regression`) or feature importances (`decision_tree`, `random_forest`, `gradient_boosting`) — that's how the proposal's "practical recommendations" objective gets fulfilled.
- Consider calibration (Brier score / calibration curve) before recommending a probability cutoff.
